In [ ]:
%pip install python-chess

     |████████████████████████████████| 134 kB 2.6 MB/s            
Note: you may need to restart the kernel to use updated packages.


In [7]:
import chess.pgn
import os

In [ ]:
filename = 'raw_db.pgn'

try:
    with open(filename, "r", encoding="utf-8") as pgn_file:
        # Read the first game from the file
        game = chess.pgn.read_game(pgn_file)

        if game:
            print(f"--- Headers for: {filename} ---")
            
            # Display specific common headers
            print(f"Event: {game.headers.get('Event', '?')}")
            print(f"Date:  {game.headers.get('Date', '?')}")
            print(f"White: {game.headers.get('White', '?')}")
            print(f"Black: {game.headers.get('Black', '?')}")
            print(f"Result: {game.headers.get('Result', '*')}")
            
            print("\n--- All Raw Headers ---")
            # Iterate through all headers found in the game
            for key, value in game.headers.items():
                print(f'[{key} "{value}"]')
        else:
            print("No games found in the file.")
            
            chess.pgn.

except FileNotFoundError:
    print(f"Error: The file '{filename}' was not found.")

--- Headers for: raw_db.pgn ---
Event: Rated Classical game
Date:  ????.??.??
White: sebaje
Black: dgsv442
Result: 0-1

--- All Raw Headers ---
[Event "Rated Classical game"]
[Site "https://lichess.org/fgvdM92j"]
[Date "????.??.??"]
[Round "?"]
[White "sebaje"]
[Black "dgsv442"]
[Result "0-1"]
[BlackElo "1605"]
[BlackRatingDiff "+10"]
[ECO "C40"]
[Opening "King's Pawn Game: Damiano Defense"]
[Termination "Normal"]
[TimeControl "600+0"]
[UTCDate "2014.04.30"]
[UTCTime "22:00:21"]
[WhiteElo "1583"]
[WhiteRatingDiff "-12"]


In [ ]:
def get_safe_elo(headers, key):
    """Attempts to parse Elo, returns None if '?' or missing."""
    val = headers.get(key)
    if val is None or val == "?":
        return None
    try:
        return int(val)
    except ValueError:
        return None

def separate_games_by_elo(input_pgn, compartments):
    compartments = sorted(compartments)
    
    with open(input_pgn, "r") as pgn_file:
        while True:
            game = chess.pgn.read_game(pgn_file)
            if game is None:
                break 

            # Safely extract ratings
            w_elo = get_safe_elo(game.headers, "WhiteElo")
            b_elo = get_safe_elo(game.headers, "BlackElo")

            # Protection: Skip game if either Elo is unknown
            if w_elo is None or b_elo is None:
                with open("games_unknown_elo.pgn", "a") as err_file:
                    err_file.write(str(game) + "\n\n")
                continue

            mean_elo = (w_elo + b_elo) / 2

            # Logic to find the bucket
            target_threshold = 0
            for limit in compartments:
                if mean_elo >= limit:
                    target_threshold = limit
                else:
                    break
            
            filename = f"games_elo_{target_threshold}.pgn"
            with open(filename, "a") as out_pgn:
                out_pgn.write(str(game) + "\n\n")

COMPARTMENTS = [800, 1000, 1400, 1800, 2000]
separate_games_by_elo(filename, COMPARTMENTS)